# OFR 资金压力：Stage1 10:32 恢复来源映射

仅使用 2017 发布后的官方当前 CSV，按第三个前序 Qlib 交易日可见，2018/2023 官方修订旧值按发布日期保守复原；2026 已知订正日期标记未知。无 SOXX 结果、拟合和账户。

In [1]:
import os,csv,json,hashlib,zipfile,xml.etree.ElementTree as ET,tarfile,tempfile,subprocess,shlex,re
from pathlib import Path
from datetime import datetime,timedelta,date
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_ofr_funding_1032_source_map_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
s=plan['source'];sc=root/s['card'];assert sha(sc/'summary.json')==s['summary_sha256'] and sha(sc/'source_result.json')==s['source_result_sha256'] and sha(sc/'revision_audit_result.json')==s['revision_audit_sha256'] and sha(sc/'publication_boundary.json')==s['publication_boundary_sha256']
assert sha(root/s['raw_csv'])==s['raw_csv_sha256'] and sha(root/s['revision_workbook'])==s['revision_workbook_sha256']
source_summary=json.loads((sc/'summary.json').read_text());assert source_summary['NAS_archive_sha256']==s['NAS_archive_sha256'] and source_summary['NAS_remote_and_scoped_restore_verified'] and not source_summary['pre_2017_current_index_PIT_qualified']
with (root/s['raw_csv']).open(newline='',encoding='utf-8-sig') as f:csv_rows=list(csv.DictReader(f))
current={pd.Timestamp(r['Date']):float(r['Funding']) for r in csv_rows};assert len(current)==len(csv_rows)>6000 and all(np.isfinite(list(current.values())))
ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'};old_by_date={};revisions=[]
with zipfile.ZipFile(root/s['revision_workbook']) as z:
 strings=[''.join(t.text or '' for t in si.findall('.//x:t',ns)) for si in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('x:si',ns)]
 for event in plan['revision_events']:
  tree=ET.fromstring(z.read('xl/worksheets/'+event['sheet']));count=0
  for row in tree.findall('.//x:sheetData/x:row',ns):
   vals={}
   for cell in row.findall('x:c',ns):
    v=cell.find('x:v',ns)
    if v is None:continue
    value=v.text
    if cell.attrib.get('t')=='s':value=strings[int(value)]
    vals[re.match(r'[A-Z]+',cell.attrib['r']).group(0)]=value
   try:
    d=pd.Timestamp((datetime(1899,12,30)+timedelta(days=float(vals[event['date_column']]))).date());old=float(vals[event['old_funding_column']]);updated=float(vals[event['updated_funding_column']])
   except (KeyError,TypeError,ValueError):continue
   assert d in current and abs(updated-current[d])<=plan['quality']['require_workbook_updated_funding_match_current_csv_abs_tol']
   assert d not in old_by_date;old_by_date[d]={'old_funding':old,'updated_funding':updated,'posted_revision_date':pd.Timestamp(event['date'])};count+=1
  revisions.append({'posted_revision_date':event['date'],'old_new_rows':count})
assert [x['old_new_rows'] for x in revisions]==[158,2,382]
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['Qlib_calendar_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-10-08',end_time=plan['action_calendar'][1],freq='day'));actions=calendar[(calendar>=pd.Timestamp(plan['action_calendar'][0]))&(calendar<=pd.Timestamp(plan['action_calendar'][1]))];source_dates=pd.DatetimeIndex(sorted(current));source_values=np.array([current[d] for d in source_dates],dtype=float)
corrected={pd.Timestamp(x) for x in ('2026-05-15','2026-05-18','2026-05-21','2026-05-22')};rows=[]
for day in actions:
 i=int(calendar.get_loc(day));cutoff=calendar[i-3] if i>=3 else pd.NaT
 j=int(source_dates.searchsorted(cutoff,side='right')-1) if pd.notna(cutoff) else -1
 obs=source_dates[j] if j>=0 else pd.NaT;age=int(i-(calendar.searchsorted(obs,side='right')-1)) if pd.notna(obs) else None
 status='known';value=np.nan;rule='current_CSV'
 if pd.isna(obs):status='unknown_no_observation'
 elif obs<pd.Timestamp(plan['quality']['source_2017_launch_boundary']):status='unknown_pre_current_FSI_launch'
 elif age>5:status='unknown_stale_source'
 elif obs in corrected:status='unknown_2026_feed_correction_first_version'
 else:
  value=source_values[j]
  if obs in old_by_date and day<=old_by_date[obs]['posted_revision_date']:
   value=old_by_date[obs]['old_funding'];rule='documented_old_before_next_session_revision'
 rows.append({'trade_date':day,'funding_stress_last_conservatively_known':float(value),'source_observation_date':obs,'source_age_Qlib_sessions':age,'source_rule':rule if status=='known' else status,'source_known':status=='known','status':status,'assumed_available_at':day+pd.Timedelta(hours=10,minutes=31),'receiver_decision_at':day+pd.Timedelta(hours=10,minutes=32)})
mapped=pd.DataFrame(rows);assert mapped.trade_date.equals(pd.Series(actions,name='trade_date')) and len(mapped)>2000 and (mapped.assumed_available_at<mapped.receiver_decision_at).all()
train=mapped.loc[mapped.trade_date.between(*[pd.Timestamp(x) for x in plan['training']])];held=mapped.loc[mapped.trade_date.between(*[pd.Timestamp(x) for x in plan['held_reused']])]
audit={'status':'OFR_funding_1032_conservative_source_map_passed' if len(train)==plan['expected_training_rows'] and len(held)==plan['expected_held_rows'] and train.source_known.mean()>=plan['quality']['minimum_train_known_fraction'] and held.source_known.mean()>=plan['quality']['minimum_held_known_fraction'] else 'OFR_funding_1032_source_map_failed','plan_sha256':sha(card/'plan.json'),'action_calendar_rows':len(actions),'train_calendar_rows':len(train),'train_known_rows':int(train.source_known.sum()),'held_calendar_rows':len(held),'held_known_rows':int(held.source_known.sum()),'train_unknown_dates':[str(d.date()) for d in train.loc[~train.source_known,'trade_date']],'held_unknown_dates':[str(d.date()) for d in held.loc[~held.source_known,'trade_date']],'revision_old_value_rows_train':int(train.source_rule.eq('documented_old_before_next_session_revision').sum()),'revision_old_value_rows_held':int(held.source_rule.eq('documented_old_before_next_session_revision').sum()),'max_source_age_Qlib_sessions':int(mapped.source_age_Qlib_sessions.dropna().max()),'revision_rows_by_posting':revisions,'historical_first_receipt_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n');assert audit['status']=='OFR_funding_1032_conservative_source_map_passed'
static=pd.concat({'feature':mapped.set_index('trade_date')[['funding_stress_last_conservatively_known']],'meta':mapped.set_index('trade_date')[['source_known','source_observation_date','source_age_Qlib_sessions']]},axis=1);static.index=pd.MultiIndex.from_arrays([mapped.trade_date,['SOXL_SOXX_CASH']*len(mapped)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['training']),'held':tuple(plan['held_reused'])});tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta']);assert len(tr)==len(train) and len(he)==len(held)
state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,dataset=dataset);rec.save_objects(artifact_path='analysis',mapped_signals=mapped,revision_old_values=pd.DataFrame([{'observation_date':d,**v} for d,v in old_by_date.items()]))
 rec.log_metrics(train_known=audit['train_known_rows'],held_known=audit['held_known_rows'],old_revision_rows_train=audit['revision_old_value_rows_train'],model_fits=0,native_accounts=0)
 with R.uri_context(str(tracking)):reread=R.get_recorder(recorder_id=rec.id,experiment_id=str(rec.experiment_id)).load_object('analysis/mapped_signals')
 assert reread.equals(mapped)
summary={'experiment':card.name,'decision':'conditional_2018plus_OFR_Funding_source_mapped_first_receipt_pending_no_fit','objective_achieved':False,'source_audit':'source_audit.json','train_calendar_rows':len(train),'train_known_rows':audit['train_known_rows'],'held_calendar_rows':len(held),'held_known_rows':audit['held_known_rows'],'historical_first_receipt_qualified':False,'new_model_fits':0,'new_native_accounts':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/mapped_signals','analysis/revision_old_values']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='ofr-funding-restore-') as td:
 archive=Path(td)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as t:
  names=[x.name for x in t.getmembers() if x.isfile()];assert any(x.endswith('/artifacts/analysis/mapped_signals') for x in names) and any(x.endswith('/artifacts/inputs/dataset') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'train_known':audit['train_known_rows'],'held_known':audit['held_known_rows'],'old_train':audit['revision_old_value_rows_train'],'old_held':audit['revision_old_value_rows_held'],'recorder_id':state['recorder_id'],'NAS_sha256':evidence['sha256']})


2026/10/23 04:54:00 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[86290:MainThread](2026-10-23 04:54:00,892) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[86290:MainThread](2026-10-23 04:54:00,894) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[86290:MainThread](2026-10-23 04:54:00,895) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


...

[86290:MainThread](2026-10-23 04:54:01,026) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_ofr_funding_1032_source_map_20261023.


[86290:MainThread](2026-10-23 04:54:01,028) INFO - qlib.workflow - [exp.py:258] - Experiment 838917316681300422 starts running ...


[86290:MainThread](2026-10-23 04:54:01,115) INFO - qlib.workflow - [recorder.py:345] - Recorder 6a55d121ea5442b4a5e558b4139e2cc8 starts running under Experiment 838917316681300422 ...


[86290:MainThread](2026-10-23 04:54:02,109) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'conditional_2018plus_OFR_Funding_source_mapped_first_receipt_pending_no_fit', 'train_known': 1059, 'held_known': 940, 'old_train': 249, 'old_held': 120, 'recorder_id': '6a55d121ea5442b4a5e558b4139e2cc8', 'NAS_sha256': 'd6d1bf043100be2202af873dbd35b44d69bf51560f88d5afe9e51fe5bf8d574b'}


# OFR Funding：Stage1 10:32 恢复动作的保守来源映射

仅针对现行 OFR FSI 发布后的 2018–2022 恢复训练窗及 2023–2026 复用历史动作窗。每个 10:32 动作日只用第三个前序 Qlib 交易日之前的最新 OFR 观测；官方工作簿列明的历史改值，在公布修订后的下一实际 Qlib 交易日前仍用旧值。未读取 SOXX 结果，0 拟合、0 账户。

Qlib 映射覆盖：训练 1059/1061，复用历史 940/944。2018-10-15/16 因该动作 Qlib 提供者在此前不足三个可读交易日而未知，未借其他日历补齐；2026-05-20/21/27/28 选中的 OFR 日期涉及官网提示的 2026 数据订正，保留未知。最大可用观测年龄 4 个 Qlib 交易日。

修订复原并非细节：训练 249 日、复用历史 120 日使用官方工作簿旧 Funding 值；工作簿新值与今日 CSV 在对应 542 条改值日期严格对齐。当前 CSV 的 2011–2017 回填仍不能作为现行 FSI 的点时训练输入。

这个状态只是**条件性研究来源**：官网现在的两个工作日数据滞后说明不认证每个历史日的首次接收、各底层数据最终性或未列明修订；今天的 2018–2026 CSV 亦不恢复所有历史网页版本。未来 0bp 实物账户与独立前向证据仍需单独通过来源时钟门。

原生 Qlib Recorder `6a55d121ea5442b4a5e558b4139e2cc8` 保存全 2006 个动作日、修订旧值和 DatasetH；NAS 归档限定回读 SHA-256 `d6d1bf043100be2202af873dbd35b44d69bf51560f88d5afe9e51fe5bf8d574b`。下一步在已封存的 0bp Tiered beta0.6 同状态恢复/HOLD 标签上，以旧 21 项为对照冻结一个 Funding 坐标的信息检验；若失败即关闭，不用符号、发布日期或阈值补救。